# RETFound for Diabetic Retinopathy: M0 (Baseline) vs M2 (Enhanced)

**M0** — RETFound ViT-L full fine-tune + Cross-Entropy  
**M2** — RETFound + LoRA + multi-scale token fusion + ordinal/focal loss

### Kaggle setup (before running)
1. Create a Notebook with **GPU** (T4/P100).
2. Add datasets (right sidebar → *Add input*):
   - **APTOS 2019**: `aptos2019-blindness-detection`
   - **RETFound weights** (recommended): upload `RETFound_cfp_weights.pth` as a private Kaggle dataset, **or** set `DOWNLOAD_WEIGHTS_FROM_HF = True` below.
3. Set `CONFIG` in the next cell (`DEMO_MODE=True` for a short smoke run).

### Local Cursor UI
You can open this `.ipynb` in Cursor and run cells locally if you have a GPU + the APTOS files.  
Kaggle cloud GPUs do **not** attach to Cursor automatically — edit here, run either locally or by uploading to Kaggle.

In [ ]:
# ========================= CONFIG =========================
class CFG:
    # Paths — auto-detect Kaggle vs local
    IS_KAGGLE = __import__("os").path.exists("/kaggle/input")

    # APTOS layout on Kaggle:
    #   /kaggle/input/aptos2019-blindness-detection/train_images/
    #   /kaggle/input/aptos2019-blindness-detection/train.csv
    DATA_DIR = "/kaggle/input/aptos2019-blindness-detection" if IS_KAGGLE else "./data/aptos2019"
    TRAIN_CSV = f"{DATA_DIR}/train.csv"
    TRAIN_IMG_DIR = f"{DATA_DIR}/train_images"

    # Put RETFound weights here (Kaggle dataset you uploaded), e.g.
    # /kaggle/input/retfound-cfp-weights/RETFound_cfp_weights.pth
    RETFOUND_WEIGHTS = (
        "/kaggle/input/retfound-cfp-weights/RETFound_cfp_weights.pth"
        if IS_KAGGLE else "./weights/RETFound_cfp_weights.pth"
    )
    DOWNLOAD_WEIGHTS_FROM_HF = True  # fallback if local path missing
    HF_REPO = "open-eye/RETFound_MAE"  # community mirror; change if your mirror differs
    HF_FILENAME = "RETFound_cfp_weights.pth"

    OUT_DIR = "/kaggle/working/outputs" if IS_KAGGLE else "./outputs"

    # Task
    NUM_CLASSES = 5  # ICDR 0-4
    IMG_SIZE = 224

    # DEMO_MODE=True → tiny subset + few epochs (pipeline check)
    # DEMO_MODE=False → full thesis run (long on Kaggle GPU)
    DEMO_MODE = True

    SEED = 42
    N_SPLITS_INFO = "70/15/15 patient-level stratified"
    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

    BATCH_SIZE = 8 if DEMO_MODE else 16
    NUM_WORKERS = 2
    EPOCHS = 2 if DEMO_MODE else 30
    DEMO_MAX_IMAGES = 400  # only used when DEMO_MODE=True

    LR_M0 = 5e-5
    LR_M2 = 1e-4
    WEIGHT_DECAY = 0.05
    WARMUP_EPOCHS = 1 if DEMO_MODE else 3

    # M2 LoRA
    LORA_R = 8
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.05
    LORA_TARGETS = ["qkv"]  # timm ViT attention

    # Multi-scale block indices (ViT-L has 24 blocks: 0..23)
    MS_BLOCKS = (7, 15, 23)

    # Loss weights for M2
    FOCAL_GAMMA = 2.0
    ORDINAL_WEIGHT = 1.0
    CE_WEIGHT = 0.5
    REFERABLE_AUX_WEIGHT = 0.3  # aux binary head (grade >= 2)

    DEVICE = "cuda"
    WHICH_MODELS = ["M0", "M2"]  # run both, or ["M0"] / ["M2"] alone

print("IS_KAGGLE:", CFG.IS_KAGGLE)
print("DEMO_MODE:", CFG.DEMO_MODE)
print("DATA_DIR:", CFG.DATA_DIR)

In [ ]:
# ========================= INSTALLS =========================
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# Avoid peft on Kaggle — it often conflicts with system torchao.
# We implement a tiny manual LoRA for timm ViT instead.
pip_install(["timm", "huggingface_hub", "scikit-learn", "opencv-python-headless"])

import os, math, json, random, time, copy
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, roc_auc_score,
    classification_report, cohen_kappa_score
)

os.makedirs(CFG.OUT_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)
CFG.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", CFG.DEVICE)
if CFG.DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
# ========================= WEIGHTS =========================
def resolve_retfound_weights(cfg=CFG) -> str:
    path = Path(cfg.RETFOUND_WEIGHTS)
    if path.is_file():
        print("Using weights:", path)
        return str(path)

    # Search common Kaggle input folders
    if cfg.IS_KAGGLE:
        for p in Path("/kaggle/input").rglob("*RETFound*cfp*.pth"):
            print("Found weights:", p)
            return str(p)
        for p in Path("/kaggle/input").rglob("*retfound*.pth"):
            print("Found weights:", p)
            return str(p)

    if not cfg.DOWNLOAD_WEIGHTS_FROM_HF:
        raise FileNotFoundError(
            f"RETFound weights not found at {cfg.RETFOUND_WEIGHTS}. "
            "Upload a Kaggle dataset or set DOWNLOAD_WEIGHTS_FROM_HF=True."
        )

    from huggingface_hub import hf_hub_download
    print(f"Downloading {cfg.HF_FILENAME} from {cfg.HF_REPO} ...")
    try:
        w = hf_hub_download(repo_id=cfg.HF_REPO, filename=cfg.HF_FILENAME)
        print("Downloaded:", w)
        return w
    except Exception as e:
        print("HF download failed:", e)
        print("Falling back to ImageNet-pretrained ViT-L (NOT true RETFound). For thesis, use official weights.")
        return ""  # empty → ImageNet init

WEIGHTS_PATH = resolve_retfound_weights()
print("WEIGHTS_PATH:", WEIGHTS_PATH or "(ImageNet fallback)")

In [ ]:
# ========================= DATA SPLITS =========================
assert Path(CFG.TRAIN_CSV).is_file(), f"Missing CSV: {CFG.TRAIN_CSV}"
assert Path(CFG.TRAIN_IMG_DIR).is_dir(), f"Missing images: {CFG.TRAIN_IMG_DIR}"

df = pd.read_csv(CFG.TRAIN_CSV)
# APTOS columns: id_code, diagnosis
df = df.rename(columns={"id_code": "image_id", "diagnosis": "label"})
df["image_id"] = df["image_id"].astype(str)
df["label"] = df["label"].astype(int)

# Resolve extension
def find_image_path(image_id, root=CFG.TRAIN_IMG_DIR):
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
        p = Path(root) / f"{image_id}{ext}"
        if p.is_file():
            return str(p)
    return None

df["path"] = df["image_id"].map(find_image_path)
missing = df["path"].isna().sum()
print(f"Rows: {len(df)} | missing files: {missing}")
df = df.dropna(subset=["path"]).reset_index(drop=True)

if CFG.DEMO_MODE:
    # Stratified tiny subset for smoke test
    df, _ = train_test_split(
        df, train_size=min(CFG.DEMO_MAX_IMAGES, len(df)),
        stratify=df["label"], random_state=CFG.SEED
    )
    df = df.reset_index(drop=True)
    print("DEMO subset size:", len(df))

# Patient-level: APTOS id_code is per-image; treat image_id as unit (no multi-image patient IDs in public CSV)
train_df, temp_df = train_test_split(
    df, test_size=(CFG.VAL_RATIO + CFG.TEST_RATIO),
    stratify=df["label"], random_state=CFG.SEED
)
rel_test = CFG.TEST_RATIO / (CFG.VAL_RATIO + CFG.TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df, test_size=rel_test,
    stratify=temp_df["label"], random_state=CFG.SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), "label counts:", d["label"].value_counts().sort_index().to_dict())

split_path = Path(CFG.OUT_DIR) / "splits.json"
split_path.write_text(json.dumps({
    "train": train_df["image_id"].tolist(),
    "val": val_df["image_id"].tolist(),
    "test": test_df["image_id"].tolist(),
    "seed": CFG.SEED,
}, indent=2))
print("Saved splits →", split_path)

In [ ]:
# ========================= DATASET / LOADERS =========================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class FundusDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, train=False, img_size=224):
        self.df = frame
        self.train = train
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.5:
            img = img.rotate(random.uniform(-15, 15), resample=Image.BILINEAR)
        # mild color jitter
        if random.random() < 0.5:
            from PIL import ImageEnhance
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.9, 1.1))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.9, 1.1))
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        if self.train:
            img = self._augment(img)
        arr = np.asarray(img).astype(np.float32) / 255.0
        arr = (arr - np.array(IMAGENET_MEAN)) / np.array(IMAGENET_STD)
        x = torch.from_numpy(arr).permute(2, 0, 1).float()
        y = int(row["label"])
        return x, y

def make_loaders(bs=CFG.BATCH_SIZE):
    train_ds = FundusDataset(train_df, train=True, img_size=CFG.IMG_SIZE)
    val_ds   = FundusDataset(val_df, train=False, img_size=CFG.IMG_SIZE)
    test_ds  = FundusDataset(test_df, train=False, img_size=CFG.IMG_SIZE)
    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=bs, shuffle=False,
                             num_workers=CFG.NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()
xb, yb = next(iter(train_loader))
print("batch", xb.shape, yb.shape, yb[:8])

In [ ]:
# ========================= LOSSES =========================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class CoralOrdinalLoss(nn.Module):
    """CORAL: K-1 binary cumulative thresholds for K ordinal classes."""
    def __init__(self, num_classes=5):
        super().__init__()
        self.num_classes = num_classes

    def levels_from_label(self, y):
        # y: (B,) long → (B, K-1) float {0,1}
        levels = torch.arange(self.num_classes - 1, device=y.device).expand(y.size(0), -1)
        return (y.unsqueeze(1) > levels).float()

    def forward(self, ordinal_logits, y):
        # ordinal_logits: (B, K-1)
        targets = self.levels_from_label(y)
        return F.binary_cross_entropy_with_logits(ordinal_logits, targets)


def coral_decode(ordinal_logits):
    probs = torch.sigmoid(ordinal_logits)
    return (probs > 0.5).sum(dim=1).long()


def class_weights_from_df(frame, n_classes=5, device="cpu"):
    counts = frame["label"].value_counts().reindex(range(n_classes), fill_value=0).values.astype(np.float32)
    counts = np.maximum(counts, 1.0)
    w = counts.sum() / (n_classes * counts)
    return torch.tensor(w, dtype=torch.float32, device=device)

CW = class_weights_from_df(train_df, CFG.NUM_CLASSES, CFG.DEVICE)
print("class weights:", CW.detach().cpu().numpy())

In [ ]:
# ========================= MODEL BUILDERS =========================
def load_vit_backbone(num_classes=0, weights_path=""):
    """ViT-Large/16 matching RETFound; num_classes=0 → features only."""
    model = timm.create_model(
        "vit_large_patch16_224",
        pretrained=(weights_path == ""),  # ImageNet if no RETFound weights
        num_classes=num_classes,
        global_pool="token",
    )
    if weights_path:
        # RETFound ckpts embed argparse.Namespace → need weights_only=False (PyTorch >= 2.6)
        # Only use this for official/trusted RETFound weights.
        try:
            ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
        except TypeError:
            ckpt = torch.load(weights_path, map_location="cpu")  # older torch
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        # Drop head / mismatched keys
        cleaned = {}
        model_sd = model.state_dict()
        for k, v in state.items():
            nk = k.replace("module.", "")
            if nk.startswith("head") or nk.startswith("fc_norm"):
                continue
            if nk in model_sd and model_sd[nk].shape == v.shape:
                cleaned[nk] = v
        missing, unexpected = model.load_state_dict(cleaned, strict=False)
        print(f"Loaded RETFound-like weights | matched={len(cleaned)} | missing≈{len(missing)} unexpected≈{len(unexpected)}")
    return model


class M0RetFound(nn.Module):
    """Baseline: full ViT-L + linear head."""
    def __init__(self, num_classes=5, weights_path=""):
        super().__init__()
        self.backbone = load_vit_backbone(num_classes=num_classes, weights_path=weights_path)

    def forward(self, x):
        return self.backbone(x)


class MultiScaleFusionHead(nn.Module):
    def __init__(self, dim=1024, num_classes=5, n_scales=3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(dim * n_scales, dim),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Linear(dim, num_classes)
        self.ordinal = nn.Linear(dim, num_classes - 1)  # CORAL
        self.referable = nn.Linear(dim, 1)

    def forward(self, feats):
        # feats: list of (B, D)
        h = self.proj(torch.cat(feats, dim=-1))
        return {
            "logits": self.classifier(h),
            "ordinal": self.ordinal(h),
            "referable": self.referable(h).squeeze(-1),
        }


class LoRALinear(nn.Module):
    """Manual LoRA on a frozen nn.Linear — avoids peft/torchao issues on Kaggle."""
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r = r
        self.scaling = alpha / r
        self.dropout = nn.Dropout(dropout) if dropout and dropout > 0 else nn.Identity()
        self.lora_A = nn.Linear(base.in_features, r, bias=False)
        self.lora_B = nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.base(x) + self.lora_B(self.lora_A(self.dropout(x))) * self.scaling


def inject_lora_timm_vit(model, r=8, alpha=16, dropout=0.05, target_names=("qkv",)):
    """Replace matching Linear modules (e.g. attn.qkv) with LoRALinear."""
    replaced = 0
    for name, module in list(model.named_modules()):
        if not any(t == name.split(".")[-1] for t in target_names):
            continue
        if not isinstance(module, nn.Linear):
            continue
        parent_path = name.rsplit(".", 1)
        if len(parent_path) == 1:
            parent, child = model, name
        else:
            parent = model.get_submodule(parent_path[0])
            child = parent_path[1]
        setattr(parent, child, LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
        replaced += 1
    return replaced


class M2RetFound(nn.Module):
    """Enhanced: LoRA ViT-L + multi-scale fusion + ordinal/referable heads."""
    def __init__(self, num_classes=5, weights_path="", ms_blocks=(7, 15, 23),
                 lora_r=8, lora_alpha=16, lora_dropout=0.05, lora_targets=None):
        super().__init__()
        self.ms_blocks = tuple(ms_blocks)
        self.backbone = load_vit_backbone(num_classes=0, weights_path=weights_path)

        for p in self.backbone.parameters():
            p.requires_grad = False

        n_lora = inject_lora_timm_vit(
            self.backbone,
            r=lora_r,
            alpha=lora_alpha,
            dropout=lora_dropout,
            target_names=tuple(lora_targets or ["qkv"]),
        )
        dim = getattr(self.backbone, "embed_dim", 1024)
        self.head = MultiScaleFusionHead(dim=dim, num_classes=num_classes, n_scales=len(self.ms_blocks))

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"LoRA layers injected: {n_lora} | trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

        self._hooks = []
        self._cache = {}
        self._register_hooks()

    def _register_hooks(self):
        blocks = self.backbone.blocks
        self._cache = {}

        def make_hook(i):
            def hook(_module, _inp, out):
                cls = out[:, 0]
                patch = out[:, 1:].mean(dim=1)
                self._cache[i] = 0.5 * (cls + patch)
            return hook

        for i in self.ms_blocks:
            self._hooks.append(blocks[i].register_forward_hook(make_hook(i)))

    def forward(self, x):
        self._cache = {}
        _ = self.backbone.forward_features(x)
        missing = [i for i in self.ms_blocks if i not in self._cache]
        if missing:
            raise RuntimeError(f"Multi-scale hooks missed blocks {missing}. Check ViT block indices.")
        feats = [self._cache[i] for i in self.ms_blocks]
        return self.head(feats)


print("Model factories ready.")

In [ ]:
# ========================= METRICS / EVAL =========================
def compute_metrics(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "qwk": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
    }
    # Referable DR: grade >= 2
    yt = (y_true >= 2).astype(int)
    yp = (y_pred >= 2).astype(int)
    out["referable_acc"] = float(accuracy_score(yt, yp))
    if y_prob is not None:
        # sum prob of classes >=2
        ref_score = y_prob[:, 2:].sum(axis=1)
        try:
            out["referable_auroc"] = float(roc_auc_score(yt, ref_score))
        except ValueError:
            out["referable_auroc"] = float("nan")
    out["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=list(range(CFG.NUM_CLASSES))).tolist()
    return out


@torch.no_grad()
def evaluate(model, loader, model_name="M0"):
    model.eval()
    ys, preds, probs = [], [], []
    for x, y in loader:
        x = x.to(CFG.DEVICE)
        out = model(x)
        if model_name == "M0":
            logits = out
            pr = torch.softmax(logits, dim=-1)
            pred = logits.argmax(dim=-1)
        else:
            logits = out["logits"]
            pr = torch.softmax(logits, dim=-1)
            # blend CE argmax with CORAL decode
            pred_ce = logits.argmax(dim=-1)
            pred_ord = coral_decode(out["ordinal"])
            pred = pred_ce  # primary head for reporting; ordinal used in training
            _ = pred_ord
        ys.append(y.numpy())
        preds.append(pred.cpu().numpy())
        probs.append(pr.cpu().numpy())
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    y_prob = np.concatenate(probs)
    return compute_metrics(y_true, y_pred, y_prob), y_true, y_pred, y_prob

In [ ]:
# ========================= TRAIN LOOPS =========================
def cosine_lr(optimizer, epoch, total_epochs, base_lr, warmup=1):
    if epoch < warmup:
        lr = base_lr * (epoch + 1) / max(1, warmup)
    else:
        t = (epoch - warmup) / max(1, total_epochs - warmup)
        lr = 0.5 * base_lr * (1 + math.cos(math.pi * t))
    for g in optimizer.param_groups:
        g["lr"] = lr
    return lr


def train_m0():
    model = M0RetFound(CFG.NUM_CLASSES, WEIGHTS_PATH).to(CFG.DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR_M0, weight_decay=CFG.WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss(weight=CW)

    best = {"qwk": -1, "path": str(Path(CFG.OUT_DIR) / "M0_best.pt")}
    history = []

    for epoch in range(CFG.EPOCHS):
        model.train()
        lr = cosine_lr(opt, epoch, CFG.EPOCHS, CFG.LR_M0, CFG.WARMUP_EPOCHS)
        losses = []
        for x, y in train_loader:
            x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        val_metrics, *_ = evaluate(model, val_loader, "M0")
        row = {"epoch": epoch, "lr": lr, "train_loss": float(np.mean(losses)), **{k: val_metrics[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}}
        history.append(row)
        print(f"[M0][{epoch}] loss={row['train_loss']:.4f} val_qwk={row['qwk']:.4f} acc={row['accuracy']:.4f} f1={row['macro_f1']:.4f}")

        if val_metrics["qwk"] > best["qwk"]:
            best["qwk"] = val_metrics["qwk"]
            torch.save({"model": model.state_dict(), "epoch": epoch, "val": val_metrics}, best["path"])

    # Load best & test
    ckpt = torch.load(best["path"], map_location=CFG.DEVICE)
    model.load_state_dict(ckpt["model"])
    test_metrics, yt, yp, ypr = evaluate(model, test_loader, "M0")
    print("[M0] TEST:", {k: test_metrics[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc", "referable_acc")})
    return model, history, test_metrics, yt, yp


def train_m2():
    model = M2RetFound(
        num_classes=CFG.NUM_CLASSES,
        weights_path=WEIGHTS_PATH,
        ms_blocks=CFG.MS_BLOCKS,
        lora_r=CFG.LORA_R,
        lora_alpha=CFG.LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
        lora_targets=CFG.LORA_TARGETS,
    ).to(CFG.DEVICE)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG.LR_M2, weight_decay=CFG.WEIGHT_DECAY)

    focal = FocalLoss(gamma=CFG.FOCAL_GAMMA, weight=CW)
    coral = CoralOrdinalLoss(CFG.NUM_CLASSES)
    bce = nn.BCEWithLogitsLoss()

    best = {"qwk": -1, "path": str(Path(CFG.OUT_DIR) / "M2_best.pt")}
    history = []

    for epoch in range(CFG.EPOCHS):
        model.train()
        lr = cosine_lr(opt, epoch, CFG.EPOCHS, CFG.LR_M2, CFG.WARMUP_EPOCHS)
        losses = []
        for x, y in train_loader:
            x, y = x.to(CFG.DEVICE), y.to(CFG.DEVICE)
            opt.zero_grad(set_to_none=True)
            out = model(x)
            loss_ce = focal(out["logits"], y)
            loss_ord = coral(out["ordinal"], y)
            ref_t = (y >= 2).float()
            loss_ref = bce(out["referable"], ref_t)
            loss = CFG.CE_WEIGHT * loss_ce + CFG.ORDINAL_WEIGHT * loss_ord + CFG.REFERABLE_AUX_WEIGHT * loss_ref
            loss.backward()
            opt.step()
            losses.append(loss.item())

        val_metrics, *_ = evaluate(model, val_loader, "M2")
        row = {"epoch": epoch, "lr": lr, "train_loss": float(np.mean(losses)), **{k: val_metrics[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc")}}
        history.append(row)
        print(f"[M2][{epoch}] loss={row['train_loss']:.4f} val_qwk={row['qwk']:.4f} acc={row['accuracy']:.4f} f1={row['macro_f1']:.4f}")

        if val_metrics["qwk"] > best["qwk"]:
            best["qwk"] = val_metrics["qwk"]
            torch.save({"model": model.state_dict(), "epoch": epoch, "val": val_metrics}, best["path"])

    ckpt = torch.load(best["path"], map_location=CFG.DEVICE)
    model.load_state_dict(ckpt["model"], strict=False)
    test_metrics, yt, yp, ypr = evaluate(model, test_loader, "M2")
    print("[M2] TEST:", {k: test_metrics[k] for k in ("accuracy", "macro_f1", "qwk", "referable_auroc", "referable_acc")})
    return model, history, test_metrics, yt, yp

print("Trainers ready.")

In [ ]:
# ========================= RUN EXPERIMENT =========================
results = {}

if "M0" in CFG.WHICH_MODELS:
    print("\n===== Training M0 (RETFound baseline) =====")
    m0, hist0, test0, yt0, yp0 = train_m0()
    results["M0"] = {"history": hist0, "test": test0}
    pd.DataFrame(hist0).to_csv(Path(CFG.OUT_DIR) / "M0_history.csv", index=False)

if "M2" in CFG.WHICH_MODELS:
    print("\n===== Training M2 (LoRA + multi-scale + ordinal/focal) =====")
    m2, hist2, test2, yt2, yp2 = train_m2()
    results["M2"] = {"history": hist2, "test": test2}
    pd.DataFrame(hist2).to_csv(Path(CFG.OUT_DIR) / "M2_history.csv", index=False)

# Comparison table
rows = []
for name, payload in results.items():
    t = payload["test"]
    rows.append({
        "model": name,
        "accuracy": t["accuracy"],
        "macro_f1": t["macro_f1"],
        "qwk": t["qwk"],
        "referable_acc": t["referable_acc"],
        "referable_auroc": t.get("referable_auroc", float("nan")),
    })

cmp = pd.DataFrame(rows)
print("\n===== TEST COMPARISON =====")
display(cmp)
cmp.to_csv(Path(CFG.OUT_DIR) / "comparison_test.csv", index=False)

# Save full metrics JSON
serializable = {
    k: {"test": {kk: vv for kk, vv in v["test"].items()}, "history": v["history"]}
    for k, v in results.items()
}
Path(CFG.OUT_DIR, "results.json").write_text(json.dumps(serializable, indent=2))
print("Wrote:", CFG.OUT_DIR)

In [ ]:
# ========================= PLOTS =========================
import matplotlib.pyplot as plt

def plot_history(results_dict):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for name, payload in results_dict.items():
        h = pd.DataFrame(payload["history"])
        ax[0].plot(h["epoch"], h["qwk"], marker="o", label=name)
        ax[1].plot(h["epoch"], h["macro_f1"], marker="o", label=name)
    ax[0].set_title("Val QWK"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].set_title("Val Macro-F1"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(Path(CFG.OUT_DIR) / "val_curves.png", dpi=150)
    plt.show()

def plot_cm(cm, title, path):
    cm = np.array(cm)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks(range(CFG.NUM_CLASSES)); ax.set_yticks(range(CFG.NUM_CLASSES))
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()

if results:
    plot_history(results)
    for name, payload in results.items():
        plot_cm(
            payload["test"]["confusion_matrix"],
            f"{name} Test Confusion Matrix",
            Path(CFG.OUT_DIR) / f"{name}_cm.png",
        )

## Next steps for the thesis run

1. Set `DEMO_MODE = False` and re-run with full APTOS.
2. Ensure **official RETFound CFP weights** are loaded (not ImageNet fallback).
3. Add an **external test** dataset (Messidor-2 / DDR) with the same preprocessing — evaluate saved `M0_best.pt` / `M2_best.pt` without retraining.
4. Optional ablations: LoRA-only, LoRA+multi-scale, full M2.
5. Run 3 seeds and report mean ± std of QWK.

### Outputs
All artifacts land in `CFG.OUT_DIR` (`/kaggle/working/outputs` on Kaggle): checkpoints, CSV comparison, confusion matrices, `results.json`.